# Kecerdasan Kolektif dan Agregasi Pendapat

**ID proyek:** `O005-LEGA-V101-PRJ01`  
**Status:** titik awal pedagogis yang ditulis secara independen.

Notebook ini menggunakan data sintetis/terbuka saja. Notebook ini **bukan** kode atau data dari makalah yang dikutip dalam bab sumber dan **bukan** klaim reproduksi hasil penelitian mana pun.


## Pertanyaan pemodelan

Kapan agregasi berbobot memperbaiki perkiraan kelompok, dan kapan konvergensi sosial hanya menyamarkan keragaman informasi?

Tujuan kerja: tetapkan sistem, jalankan eksperimen deterministik, periksa invarian, visualisasikan perilaku, lalu kritik kecukupan model.


In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

SEED = 2026082201
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=6, suppress=True)


## Struktur dan asumsi

Agen mengamati sinyal skalar yang sama dengan simpangan baku berbeda. Bobot menyatakan presisi yang diketahui, sedangkan pembaruan sosial mencampurkan pendapat pribadi dan rerata kelompok.

Semua skala dan parameter di notebook ini bersifat ilustratif. Ubah satu asumsi pada satu waktu dan catat dampaknya pada keluaran serta invarian.


In [ ]:
n_agents = 81
true_signal = 0.65
precision = rng.uniform(2.0, 8.0, n_agents)
sigma = 1.0 / np.sqrt(precision)
initial_opinions = true_signal + rng.normal(0.0, sigma)
weights = precision / precision.sum()
collective_estimate = float(np.sum(weights * initial_opinions))

opinions = initial_opinions.copy()
variance_history = [float(np.var(opinions))]
for _ in range(16):
    group_mean = float(np.sum(weights * opinions))
    opinions = 0.68 * opinions + 0.32 * group_mean
    variance_history.append(float(np.var(opinions)))
final_opinions = opinions.copy()


## Pemeriksaan numerik

Pemeriksaan berikut sengaja berada di dalam notebook: eksekusi berhenti bila suatu invarian dasar gagal. Ini bukan bukti bahwa model benar; ini hanya bukti bahwa implementasi memenuhi kontrak numerik terbatasnya.


In [ ]:
assert np.isfinite(initial_opinions).all() and np.isfinite(weights).all()
assert variance_history[-1] < 0.01 * variance_history[0]
assert abs(collective_estimate - true_signal) < 0.20
np.testing.assert_allclose(weights.sum(), 1.0, atol=1e-14)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
axes[0].hist(initial_opinions, bins=14, alpha=0.75, label="awal")
axes[0].hist(final_opinions, bins=14, alpha=0.75, label="setelah pembaruan")
axes[0].axvline(true_signal, color="black", linestyle="--", label="sinyal")
axes[0].set(xlabel="pendapat", ylabel="jumlah agen", title="Sebaran pendapat")
axes[0].legend(fontsize=8)
axes[1].plot(variance_history, marker="o", markersize=3)
axes[1].set(xlabel="langkah sosial", ylabel="ragam", title="Konvergensi kelompok")
fig.tight_layout()
plt.show()
plt.close(fig)


## Validasi, identifikasi, dan keterbatasan

Keterbatasan awal: Model tidak memuat jaringan kepercayaan, bias sistematis bersama, perilaku strategis, atau cara realistis untuk mengetahui presisi setiap agen.

Jawab sebelum menafsirkan gambar:

1. Besaran apa yang benar-benar dapat diamati, dan bagaimana galat pengukurannya dimodelkan?
2. Parameter mana yang dapat diidentifikasi dari keluaran tersebut? Tunjukkan dengan profil galat, pemisahan latih/uji, atau eksperimen sensitivitas.
3. Invarian atau pola kualitatif apa yang harus tetap benar ketika ukuran langkah, benih acak, atau resolusi diubah?
4. Temukan satu skenario kegagalan model dan jelaskan data tambahan yang diperlukan untuk membedakannya dari model alternatif.


## Daftar periksa reproduksibilitas

- [ ] Gunakan CPython dan versi paket tepat seperti `requirements.lock`.
- [ ] Jalankan ulang dari kernel kosong tanpa jaringan.
- [ ] Pertahankan nilai `SEED` (benih acak), lalu ulangi dengan sedikitnya lima benih acak lain dan laporkan variasinya.
- [ ] Catat setiap perubahan parameter, persamaan, toleransi, serta pembagian data.
- [ ] Pastikan semua uji lulus dan jelaskan mengapa tiap uji relevan.
- [ ] Simpan hasil turunan di luar notebook sumber; notebook distribusi harus tetap tanpa keluaran tersimpan.
- [ ] Bedakan hasil simulasi, data sintetis, dan klaim empiris secara eksplisit.
